1. Evaluate Yolo Result 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load YOLO training results
# results_path = "/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/yolo_train_part/results.csv" 
results_path = "/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/train_SVI_Yolo/results.csv"  
df = pd.read_csv(results_path)
df

In [ ]:
df.columns

In [ ]:
# Set figure size
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training Losses (Box, Class, DFL)
axes[0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss')
axes[0].plot(df['epoch'], df['train/cls_loss'], label='Train Class Loss')
axes[0].plot(df['epoch'], df['train/dfl_loss'], label='Train DFL Loss')
axes[0].set_title("Training Losses")
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Normalize Validation Losses to match Training Scale
val_box_loss = df['val/box_loss'] / df['val/box_loss'].max() * df['train/box_loss'].max()
val_cls_loss = df['val/cls_loss'] / df['val/cls_loss'].max() * df['train/cls_loss'].max()
val_dfl_loss = df['val/dfl_loss'] / df['val/dfl_loss'].max() * df['train/dfl_loss'].max()

# Validation Losses (Rescaled)
axes[1].plot(df['epoch'], val_box_loss, label='Val Box Loss (Rescaled)', linestyle="dashed")
axes[1].plot(df['epoch'], val_cls_loss, label='Val Class Loss (Rescaled)', linestyle="dashed")
axes[1].plot(df['epoch'], val_dfl_loss, label='Val DFL Loss (Rescaled)', linestyle="dashed")
axes[1].set_title("Validation Losses (Rescaled)")
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Loss (Scaled)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

1. Training Losses (Left Plot)  
Steady decline for box loss (blue), class loss (orange), and DFL loss (green).  
The model continues learning even at epoch 200, though the improvement is slower.  
No sudden spikes, indicating a stable training process.  
✅ Good signs:  

The loss decreases consistently.  
No early stagnation or divergence, meaning the model isn't overfitting to training data.  
2. Validation Losses (Right Plot, Rescaled)  
The validation losses fluctuate but remain stable after the first 20-30 epochs.  
Box Loss (Blue) and DFL Loss (Green) are relatively high and still fluctuating slightly.  
Class Loss (Orange) is much lower, meaning classification accuracy is stable.  
🔍 Key observations:  

The spikes in validation box loss around epochs ~50, 100, and 150 suggest that some batches might be harder to detect.  
The general trend matches the training loss, which is a good sign (no extreme overfitting).  
The validation class loss is close to zero, suggesting YOLO is classifying objects well, but might struggle with localization (bounding box accuracy).  
3. What This Means for Your Model  
✅ No Overfitting: Validation loss does not explode or diverge from training loss.  
✅ Generalization Looks Good: The validation loss does not increase after stabilizing.  
❗ Bounding Box Refinement Needed: Spikes in validation box loss suggest the model might not always place bounding boxes correctly.  

4. How to Improve the Model?  
Option 1: Improve Bounding Box Accuracy  
Increase dataset diversity (especially for edge cases where objects are harder to detect).  
Fine-tune anchor boxes if using YOLO anchors.  
Option 2: Reduce Spikes in Validation Loss  
Increase validation batch size to smooth out loss fluctuations.  
Use more aggressive augmentation (e.g., MixUp, CutMix) to improve robustness.  
Option 3: Tune the Learning Rate  
If the learning rate is too high, it may be causing fluctuations.  
Plot the learning rate curve to check if adjustments are needed.  
Final Verdict  
🔹 Overall, your model is learning well and generalizing effectively.  
🔹 The main issue is the bounding box loss, which could be improved through better training data or hyperparameter tuning.  
🔹 The next step is to analyze the learning rate schedule and confusion matrix to refine detection accuracy.  

In [ ]:
# Set figure size
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Precision
axes[0].plot(df['epoch'], df['metrics/precision(B)'], label="Precision", color="blue")
axes[0].set_title("Precision")
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Score")
axes[0].legend()
axes[0].grid(True)

# Recall
axes[1].plot(df['epoch'], df['metrics/recall(B)'], label="Recall", color="green")
axes[1].set_title("Recall")
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Score")
axes[1].legend()
axes[1].grid(True)

# mAP (Mean Average Precision)
axes[2].plot(df['epoch'], df['metrics/mAP50(B)'], label="mAP@50", color="red")
axes[2].plot(df['epoch'], df['metrics/mAP50-95(B)'], label="mAP@50-95", color="orange")
axes[2].set_title("mAP Scores")
axes[2].set_xlabel("Epochs")
axes[2].set_ylabel("Score")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Set figure size
plt.figure(figsize=(8, 5))

# Learning Rates
plt.plot(df['epoch'], df['lr/pg0'], label="lr/pg0")
plt.plot(df['epoch'], df['lr/pg1'], label="lr/pg1")
plt.plot(df['epoch'], df['lr/pg2'], label="lr/pg2")

# Formatting
plt.xlabel("Epochs")
plt.ylabel("Learning Rate")
plt.title("Learning Rate Schedule")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
best_epoch = df.loc[df['metrics/mAP50(B)'].idxmax()]
print("Best Epoch:")
print(best_epoch)

In [ ]:
# import os

# # Define paths for the two image folders
# folder_1 = "/Users/wenlanzhang/Downloads/PhD_UCL/Data/GoogleStreetView/Train/Train0315_695/images/"  # First folder
# folder_2 = "/Users/wenlanzhang/Downloads/PhD_UCL/Data/GoogleStreetView/mix"  # Second folder

# # List all image files (remove extensions for comparison)
# images_1 = {f.replace(".jpg", "") for f in os.listdir(folder_1) if f.endswith(".jpg")}
# images_2 = {f.replace(".jpg", "") for f in os.listdir(folder_2) if f.endswith(".jpg")}

# # Find missing images
# missing_in_folder_1 = images_2 - images_1  # Images in folder_2 but not in folder_1
# missing_in_folder_2 = images_1 - images_2  # Images in folder_1 but not in folder_2

# # Print results
# print(f"📸 Total images in {folder_1}: {len(images_1)}")
# print(f"📸 Total images in {folder_2}: {len(images_2)}")

# print(f"🚫 Missing in {folder_1} ({len(missing_in_folder_1)}): {missing_in_folder_1}")
# print(f"🚫 Missing in {folder_2} ({len(missing_in_folder_2)}): {missing_in_folder_2}")
